In [74]:
import pandas as pd

data = pd.read_csv('matches.csv')
data["rank_a"] = data["rank_a"].astype("Int64")


def cleanup(df): 
    remove_duplicate_data = df.copy()
    remove_duplicate_data = remove_duplicate_data.drop_duplicates().reset_index(drop=True)

    return remove_duplicate_data


remove_duplicate_data = cleanup(data)
remove_duplicate_data["rank_a"] = remove_duplicate_data["rank_a"].astype("Int64")

print("\n\n\nНовая таблица без дубликатов матчей: ")
print(remove_duplicate_data.to_string())

# remove_duplicate_data.to_csv('matches_after_proc.csv', index=False)

# clean_df = data.loc[~(data["rank_a"].isna())].drop_duplicates(subset="match_id").reset_index(drop=True)
# print("\n\n\nПолностью чистая от багов таблица")
# print(clean_df.to_string())

date_of_cut = "2026-10-06"


def separation_by_date(data: pd.DataFrame, date):
    before_date = data.loc[data["date"] < date]
    after_date = data.loc[data["date"] >= date]

    return before_date, after_date


history, predict_date = separation_by_date(remove_duplicate_data, date_of_cut)


after_data = history.dropna(subset=["rank_a", "rank_b"])
after_data = after_data.loc[after_data["rank_a"] != after_data["rank_b"]]


def counting_total_number_of_lines_and_favorites_wins(data:pd.DataFrame) -> tuple[int, int]:
    n = data.shape[0]
    k = ((data["rank_a"] > data["rank_b"]) & (data["sets_a"] < data["sets_b"])).sum()
    k += ((data["rank_a"] < data["rank_b"]) & (data["sets_a"] > data["sets_b"])).sum()

    return n, k


n, k = counting_total_number_of_lines_and_favorites_wins(after_data)
p_fav = k / n

print("\nДля дат до 2026-10-06:")
print(f"n = {n}, k = {k}, p_fav = {p_fav}")


def calculating_prob_of_player_A_winning(match, p_fav: float) -> float:
    return p_fav if match.rank_a < match.rank_b else 1 - p_fav


def calculating_statistics_wins_of_player_A(data: pd.DataFrame, p_fav: float, count_MSE: bool) -> bool:
    if (count_MSE):
        squared_error_sum = 0.0
        evaluated_count = 0
        
    for match in data.itertuples(index=False):
        if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
                print(f"--SKIP--")
                continue

        p_a = calculating_prob_of_player_A_winning(match, p_fav)
        fact = 1 if match.sets_a > match.sets_b else 0
        if (count_MSE):
            squared_error_sum += (p_a - fact)**2
            evaluated_count += 1

        print(f"match_id={match.match_id}, p_a={p_a}, fact={fact}")

    if (count_MSE):
        return squared_error_sum / evaluated_count
    return None


print(f"MSE: {calculating_statistics_wins_of_player_A(predict_date, p_fav, False)}")





Новая таблица без дубликатов матчей: 
   match_id        date surface player_a player_b  rank_a  rank_b  sets_a  sets_b
0         1  2026-10-01    Hard    Orlov    Costa      18      42       2       0
1         2  2026-10-02    Clay     Vega      Kim      55      24       2       1
2         3  2026-10-03    Hard      Kim    Orlov      24      18       0       2
3         4  2026-10-04    Clay    Costa     Vega      42      55       1       2
4         5  2026-10-05    Hard     Vega    Orlov    <NA>      18       0       2
5         6  2026-10-06    Hard    Costa      Kim      42      24       1       2
6         7  2026-10-07    Clay    Orlov     Vega      18      55       1       2
7         8  2026-10-08    Hard      Kim    Costa      24      42       2       1

Для дат до 2026-10-06:
n = 4, k = 2, p_fav = 0.5
match_id=6, p_a=0.5, fact=0
match_id=7, p_a=0.5, fact=0
match_id=8, p_a=0.5, fact=1
MSE: None


<div style="border-left: 2px solid #f65c5c; padding: 8px 12px; max-width: 120ch;">
    Мне кажется, что доля побед фаворита по группам будет более объективной оценкой. Причем, чем больше будет различаться rank двух игроков, я тем больше будет точность вероятности победы фаворита. Объясняется это тем, что существенная разница в rank соперников отображает разницу уровня игроков. От уровня игроков напрямую зависит результат: чем выше уровень мастерства игрока, тем больше шансов в действительности победить более слабого соперника.Больше всего "неточностей", т.е. когда вероятность победы фаворита высока, а побеждает андердог, будет происходить, когда rank соперников несущественно различается. Нужно только грамотно определить порог этой "неточности", когда мы можем доверять вероятности победы фаворита
</div>

<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
Пока что короче остановлюсь на gap=16.
В матчах с разрывом в рейтинге 16 мест и больше фаворит побеждает чаще, чем в матчах с разрывом 1–15 мест.
</div>

In [50]:
import pandas as pd
import numpy as np

data = pd.read_csv("matches_after_proc.csv")
data = cleanup(data)
data["rank_a"] = data["rank_a"].astype("Int64")


def define_gap_and_gap_group(data: pd.DataFrame, porog: int) -> pd.DataFrame:
    data["gap"] = (data["rank_a"] - data["rank_b"]).abs()
    data["gap_group"] = np.select(
        [data["gap"].isna(), (data["gap"] < porog).to_numpy(dtype=bool, na_value=False)],
        ["--SKIP--", "closest"],
        default="farthest"
    )

    return data

data = define_gap_and_gap_group(data, 16)

print(data.to_string())

    match_id        date surface player_a player_b  rank_a  rank_b  sets_a  sets_b   gap gap_group
0          1  2026-10-01    Hard    Orlov    Costa      18      42       2       0    24  farthest
1          2  2026-10-02    Clay     Vega      Kim      55      24       2       1    31  farthest
2          3  2026-10-03    Hard      Kim    Orlov      24      18       0       2     6   closest
3          4  2026-10-04    Clay    Costa     Vega      42      55       1       2    13   closest
4          5  2026-10-05    Hard     Vega    Orlov    <NA>      18       0       2  <NA>  --SKIP--
5          6  2026-10-06    Hard    Costa      Kim      42      24       1       2    18  farthest
6          7  2026-10-07    Clay    Orlov     Vega      18      55       1       2    37  farthest
7          8  2026-10-08    Hard      Kim    Costa      24      42       2       1    18  farthest
8          9  2026-10-09    Hard      Kim    Orlov      24      18       2       1     6   closest
9         

In [54]:
"""
    Предикт победы фаворита без учета его gap/gap_group.
    В будущем сравню его с предиктом, который учитывает gap/gap_group
"""

date_for_cut = "2026-10-17"

history, predict_date = separation_by_date(data, date_for_cut)

n_without_gap, k_without_gap = counting_total_number_of_lines_and_favorites_wins(history)

fav_p_without_gap = k_without_gap / n_without_gap
print(f"n_without_gap={n_without_gap}, k_without_gap={k_without_gap}, fav_p_without_gap={round(fav_p_without_gap, 2)}")

print(f"MSE={calculating_statistics_wins_of_player_A(predict_date, fav_p_without_gap, True)}")


n_without_gap=16, k_without_gap=9, fav_p_without_gap=0.56
match_id=17, p_a=0.5625, fact=1
match_id=18, p_a=0.4375, fact=0
match_id=19, p_a=0.5625, fact=0
match_id=20, p_a=0.5625, fact=1
MSE=0.22265625


In [ ]:
"""
    Предикт победы фаворита с учетом его gap/gap_group
    В будущем сравню его с предиктом, который не учитывает gap/gap_group
"""

date_for_cut = "2026-10-17"

history, predict_date = separation_by_date(data, date_for_cut)

n_closest_group = 0
k_closest_group = 0
n_farthest_group = 0
k_farthest_group = 0

for match in history.itertuples():
    if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
        print("--SKIP--")
        continue

    if (match.gap_group == 'closest'):
        n_closest_group += 1
        k_closest_group += (match.rank_a < match.rank_b) & (match.sets_a > match.sets_b) | \
                           (match.rank_b < match.rank_a) & (match.sets_b > match.sets_a)
    else:    
        n_farthest_group += 1
        k_farthest_group += (match.rank_a < match.rank_b) & (match.sets_a > match.sets_b) | \
                            (match.rank_b < match.rank_a) & (match.sets_b > match.sets_a)

p_fav_closest_group = k_closest_group / n_closest_group
p_fav_farthest_group = k_farthest_group / n_farthest_group

print(f"n_closest_group={n_closest_group}, k_closest_group={k_closest_group}, p_fav_closest_group={round(p_fav_closest_group, 2)}")
print(f"n_farthest_group={n_farthest_group}, k_farthest_group={k_farthest_group}, p_fav_farthest_group={round(p_fav_farthest_group, 2)}")

print("\n\nПредикт на 4 матча с учетом gap/gap_group:")

def calculating_statistics_wins_of_player_A_with_gap(data: pd.DataFrame, p_fav_closest_group: float, p_fav_farthest_group, count_MSE: bool):
    if count_MSE:
        squared_error_sum = 0.0
        evaluated_count = 0

    for match in predict_date.itertuples():
        if (pd.isna(match.rank_a) or pd.isna(match.rank_b) or match.rank_a == match.rank_b):
            print("--SKIP--")
            continue

        if (match.gap_group == 'closest'):
            p_A = calculating_prob_of_player_A_winning(match, p_fav_closest_group)
        else:     
            p_A = calculating_prob_of_player_A_winning(match, p_fav_farthest_group)

        fact = 1 if match.sets_a > match.sets_b else 0
        if count_MSE:
            squared_error_sum += (p_A - fact)**2
            evaluated_count += 1

        print(f"match_id={match.match_id}, group={match.gap_group}, p_A={round(p_A, 2)}, fact={fact}")

    if count_MSE:
        return squared_error_sum / evaluated_count
    return None
        
print(f"\nСреднеквадратичная ошибка предикта с учета gap: {round(calculating_statistics_wins_of_player_A_with_gap(predict_date, p_fav_closest_group, p_fav_farthest_group, True), 4)}")

--SKIP--
n_closest_group=6, k_closest_group=2, p_fav_closest_group=0.33
n_farthest_group=9, k_farthest_group=7, p_fav_farthest_group=0.78


Предикт на 4 матча с учетом gap/gap_group:
match_id=17, group=closest, p_A=0.33, fact=1
match_id=18, group=farthest, p_A=0.22, fact=0
match_id=19, group=closest, p_A=0.33, fact=0
match_id=20, group=farthest, p_A=0.78, fact=1

Среднеквадратичная ошибка предикта с учета gap: 0.1636


<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
Понял, что пора переходить на реальные данные
</div>

In [67]:
"""
    Из открытых источников нашел данные реальных матчей за 2022 и 2023 года.
    Подчищаю и вывожу только нужные столбцы
"""

import pandas as pd


data2022 = pd.read_csv("atp_matches_2022.csv")
data2023 = pd.read_csv("atp_matches_2023.csv")
data2022["season"] = 2022
data2023["season"] = 2023

data = pd.concat([data2022, data2023], ignore_index=True)
counts = data.groupby("season").size()

rows_before_drop_duplicates = len(data)
data = data.drop_duplicates().reset_index(drop=False)
rows_after_drop_duplicates = len(data)

columns = ["tourney_id", "match_num", "score", "winner_rank", "loser_rank", "season"]
data_before_proc = data
data = data.loc[:, columns]

duplicated_rows = data.duplicated(subset=["tourney_id", "match_num"], keep=False)
if duplicated_rows.any():
    print(data.loc[duplicated_rows].to_string())

print(data.head(10).to_string(index=False))

tourney_id  match_num             score  winner_rank  loser_rank  season
 2022-8888        300        7-6(3) 6-3         11.0        19.0    2022
 2022-8888        299           6-4 6-3         14.0        20.0    2022
 2022-8888        298 7-6(6) 2-6 7-6(5)         19.0         9.0    2022
 2022-8888        297           6-2 6-1         20.0       860.0    2022
 2022-8888        296           6-4 6-0          2.0        11.0    2022
 2022-8888        295       6-4 5-7 6-4         14.0       167.0    2022
 2022-8888        294           6-3 7-5          8.0        33.0    2022
 2022-8888        293           6-2 7-5         42.0       345.0    2022
 2022-8888        292   4-6 6-4 3-0 RET         17.0        33.0    2022
 2022-8888        291    6-4 3-6 7-6(5)         42.0       139.0    2022


In [6]:
data.loc[:, ["tourney_id", "match_num", "score", "winner_rank", "loser_rank"]].to_csv("atp_matches_2022_2023_drop_dup.csv", index=False)

In [72]:
"""
    Убираю все пустые/некорректные поля + когда матч закончился из-за (RET)/(W/O)/(DIF)
"""

print(data_before_proc.shape[0])


def cleanup_uncorrected_values(data: pd.DataFrame):
    data = data.loc[~((data_before_proc["winner_rank"] <= 0) | 
                      (data_before_proc["winner_rank"].isna()) | 
                      (data_before_proc["loser_rank"] <= 0) | 
                      (data_before_proc["loser_rank"].isna()))]
    return data


data_before_proc = cleanup_uncorrected_values(data_before_proc)
print(data_before_proc.shape[0])

data = data_before_proc
data = data.loc[:, columns]


def cleanup_uncorrented_score(data: pd.DataFrame):
    empties = data["score"].isna()
    has_letters = data["score"].str.contains(r"[A-Za-z]")
    return data.loc[~empties].loc[~has_letters]


data = cleanup_uncorrented_score(data)
print(f"После фильтра по счёту: {len(data)}")

equal_ranks = data["winner_rank"].eq(data["loser_rank"])
print(f"Матчей с равными рейтингами: {equal_ranks.sum()}")

data = data.loc[~equal_ranks].copy()

print(f"Осталось матчей: {len(data)}")
print(data.shape[0])
print(data.head(10).to_string())


5833
5833
После фильтра по счёту: 5648
Матчей с равными рейтингами: 0
Осталось матчей: 5648
5648
   tourney_id  match_num              score  winner_rank  loser_rank  season
0   2022-8888        300         7-6(3) 6-3         11.0        19.0    2022
1   2022-8888        299            6-4 6-3         14.0        20.0    2022
2   2022-8888        298  7-6(6) 2-6 7-6(5)         19.0         9.0    2022
3   2022-8888        297            6-2 6-1         20.0       860.0    2022
4   2022-8888        296            6-4 6-0          2.0        11.0    2022
5   2022-8888        295        6-4 5-7 6-4         14.0       167.0    2022
6   2022-8888        294            6-3 7-5          8.0        33.0    2022
7   2022-8888        293            6-2 7-5         42.0       345.0    2022
9   2022-8888        291     6-4 3-6 7-6(5)         42.0       139.0    2022
10  2022-8888        290            6-1 6-4         19.0        33.0    2022


In [ ]:
"""
    Вывод вероятности победы фаворита по двум группам.
    В конец добавил разницу, чтобы было нагляднее
"""

def define_gap_and_gap_group(data: pd.DataFrame, porog: int) -> pd.DataFrame:
    data["gap"] = (data["winner_rank"] - data["loser_rank"]).abs()
    data["gap_group"] = np.select(
        [data["gap"].isna(), (data["gap"] < porog).to_numpy(dtype=bool, na_value=False)],
        ["--SKIP--", "closest"],
        default="farthest"
    )

    return data

data = define_gap_and_gap_group(data, 16)

def define_fav_won(data: pd.DataFrame):
    data["fav_won"] = np.where(data["winner_rank"] < data["loser_rank"], 1, 0)
    return data

data = define_fav_won(data)

n = data.groupby("gap_group").size()
k = data.groupby("gap_group")["fav_won"].sum()
p = k / n

summary = pd.DataFrame({"n": n, "k": k, "p": p})
print(summary.round(4))

diff = 100 * (p.loc["closest"] - p.loc["farthest"])
print(f"Разница: {diff:.2f} п. п.")

              n     k       p
gap_group                    
closest    1168   659  0.5642
farthest   4480  2974  0.6638
Разница: -9.96 п. п.


<div style="border-left: 2px solid #88f65c; padding: 8px 12px; max-width: 120ch;">
В сезонах 2022–2023 фаворит по рейтингу выигрывал на 9,96 процентов чаще при разрыве от 16 мест. На сезоне 2024 буду проверять, улучшает ли учёт gap прогноз по MSE(Brier score)
</dic>

In [9]:
data2024 = pd.read_csv("atp_matches_2024.csv")

In [76]:
print(f"Количество строк до обработки: {data2024.shape[0]}")
rows_before_drop_duplicates_2024 = len(data2024)


def cleanup_uncorrected_values(data: pd.DataFrame):
    valid_ranks = (data["winner_rank"] > 0) & (data["loser_rank"] > 0)
    return data.loc[valid_ranks]


data2024 = cleanup(data2024)

rows_after_drop_duplicates_2024 = len(data2024)
data2024["season"] = 2024

print(f"Количество строк до удаления дубликатов: {data2024.shape[0]}")

columns = ["tourney_id", "match_num", "score", "winner_rank", "loser_rank", "season"]
data2024 = data2024.loc[:, columns]

data2024 = cleanup_uncorrected_values(data2024)

equal_ranks = data2024["winner_rank"].eq(data2024["loser_rank"])
data2024 = data2024.loc[~equal_ranks]

data2024 = cleanup_uncorrented_score(data2024)
print(f"Количество строк после обработки: {data2024.shape[0]}")

data2024["gap_group"] = np.where((data2024["winner_rank"] - data2024["loser_rank"]).abs() < 16, "closest", "farthest")
data2024["fav_won"] = np.where(data2024["winner_rank"] < data2024["loser_rank"], 1, 0)

data2024["p_baseline"] = summary["k"].sum() / summary["n"].sum()
data2024["p_gap"] = np.where(data2024["gap_group"].eq("closest"), summary.loc["closest","k"] / summary.loc["closest","n"], \
                                                                  summary.loc["farthest","k"] / summary.loc["farthest","n"])
print(data2024.head(5).to_string())

brier_baseline = ((data2024["fav_won"] - data2024["p_baseline"])**2).mean()
brier_gap = ((data2024["p_gap"] - data2024["fav_won"])**2).mean()

print(f"brier_baseline={brier_baseline:.8f}, brier_gap={brier_gap:.8f}, diff={(brier_baseline - brier_gap):.8f}")

improvement = (data2024["p_baseline"] - data2024["fav_won"]) ** 2 - (data2024["p_gap"] - data2024["fav_won"]) ** 2
print(improvement.groupby(data2024["gap_group"]).agg(["size", "mean"]))

Количество строк до обработки: 2927
Количество строк до удаления дубликатов: 2927
Количество строк после обработки: 2927
  tourney_id  match_num       score  winner_rank  loser_rank  season gap_group  fav_won  p_baseline     p_gap
0  2024-0339        300  7-6(5) 6-4         14.0         8.0    2024   closest        0    0.643237  0.564212
1  2024-0339        299  6-4 7-6(0)          8.0        39.0    2024  farthest        1    0.643237  0.663839
2  2024-0339        298     6-3 7-5         14.0        55.0    2024  farthest        1    0.643237  0.663839
3  2024-0339        297  6-2 7-6(6)          8.0       116.0    2024  farthest        1    0.643237  0.663839
4  2024-0339        296  7-6(4) 6-2         39.0        44.0    2024   closest        1    0.643237  0.564212
brier_baseline=0.23043788, brier_gap=0.22981888, diff=0.00061900
           size      mean
gap_group                
closest     595  0.002979
farthest   2332  0.000017


<div style="border-left: 2px solid #d46e20; padding: 8px 12px; max-width: 120ch;">
На самом деле достаточно грустно, что всего в ~0,3% произошло улучшение(на closest группах). На farthest группах вообще ~0.0002%. Короче, грусть
</div>

In [11]:
data2024.to_csv("atp_matches_2024_drop_dup.csv", index=False)